In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import roc_curve, roc_auc_score
#Load estimation dataset
mort_est_df = pd.read_csv('mort_for_estimation.csv', header=0, parse_dates=['ORIG_DATE1', 'DATE_NOW'])

mort_est_df.sort_values(['LOAN_ID', 'LOAN_AGE'])
mort_start_df= mort_est_df.drop_duplicates('LOAN_ID')
mort_est_df['refi_benefit_now']= mort_est_df['refi_benefit_now'] * -1
mort_est_df['refi_benefit_past'] = mort_est_df['refi_benefit_past'] * -1

In [ ]:
mort_start_df[['ORIG_RATE', 'ORIG_UPB', 'OLTV', 'CSCORE_MAX', 'coop_condo_dummy', 'SATO']].describe()

In [ ]:
mort_start_df["ORIG_DATE1"].value_counts().sort_index().reset_index()

In [ ]:
#Origination Date
plt.hist(mort_start_df["ORIG_DATE1"], color='skyblue', edgecolor='black')
# Adding labels and title
plt.xlabel('Origination Date)')
plt.ylabel('Frequency')
plt.title('Origination Date')

# Display the plot
plt.show()

In [ ]:
#Origination UPB
plt.hist(mort_start_df["ORIG_UPB"]/1000, bins=30, color='skyblue', edgecolor='black')
# Adding labels and title
plt.xlabel('Original UPB ' + '($1,000s)')
plt.ylabel('Frequency')
plt.title('Original UPB ' +'($1,000s)')

# Display the plot
plt.show()

In [ ]:
plt.hist(mort_start_df["ORIG_RATE"], bins=60, color='skyblue', edgecolor='black')

# Adding labels and title
plt.xlabel('Original Interest Rate')
plt.ylabel('Frequency')
plt.title('Original Interest Rate')

# Display the plot
plt.show()

In [ ]:
plt.hist(mort_est_df["OLTV"], bins=30, color='skyblue', edgecolor='black')

# Adding labels and title
plt.xlabel('Original LTV')
plt.ylabel('Frequency')
plt.title('Original LTV')

# Display the plot
plt.show()

In [ ]:
plt.hist(mort_start_df["CSCORE_MAX"], bins=60, color='skyblue', edgecolor='black')

# Adding labels and title
plt.xlabel('Borrower Credit Score')
plt.ylabel('Frequency')
plt.title('Borrower Credit Score')

# Display the plot
plt.show()

In [ ]:
plt.hist(mort_start_df["SATO"], bins=60, color='skyblue', edgecolor='black')

# Adding labels and title
plt.xlabel('Interest Rate Spread at Time of Origination (SATO)')
plt.ylabel('Frequency')
plt.title('Interest Rate Spread at Time of Origination (SATO)')

# Display the plot
plt.show()

In [ ]:
loan_count_df =mort_est_df["LOAN_AGE"].value_counts().reset_index()
loan_count_df = loan_count_df[loan_count_df['LOAN_AGE']>0]
loan_count_df = loan_count_df.sort_values(by='LOAN_AGE')

plt.figure(figsize=(10, 6))
sns.lineplot(data=loan_count_df, x="LOAN_AGE", y="count", marker='o', label='Loan Count')
plt.title("Loan Count by Loan Age")
plt.xlabel("Loan Age (Months)")
plt.ylabel("Count of Loans")

In [ ]:
# Calculate home price appreciation by loan_age
medians = mort_est_df.groupby('LOAN_AGE')['hpi_growth'].median()
p10 = mort_est_df.groupby('LOAN_AGE')['hpi_growth'].quantile(0.1)
p90 = mort_est_df.groupby('LOAN_AGE')['hpi_growth'].quantile(0.9)
plt.figure(figsize=(10, 6))
plt.plot(medians.index, medians.values, marker='o', linewidth=2, label ='Median')
plt.plot(p10.index, p10.values, linestyle='--', color='orange', label='10th Percentile')
plt.plot(p90.index, p90.values, linestyle='--', color='green', label='90th Percentile')
plt.xlabel('Loan Age')
plt.ylabel('Cumulative HPI Growth')
plt.title('Cumulative HPI Growth by Loan Age')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
mort_rates_df = pd.read_csv("~/mortgage_data/MORTGAGE30US.csv", sep=',', header=0)
mort_rates_df['mort_rate_date'] = pd.to_datetime(mort_rates_df['observation_date'])
mort_rates_df = mort_rates_df.drop(columns=['observation_date'])
mort_rates_df['MORTGAGE30US'] = mort_rates_df['MORTGAGE30US'] / 100
mort_rates_df = mort_rates_df[mort_rates_df['mort_rate_date'] > '2013-01-01'].copy()
mort_rates_df = mort_rates_df.sort_values('mort_rate_date').reset_index(drop=True)

plt.figure(figsize=(10, 6))
sns.lineplot(data=mort_rates_df, x="mort_rate_date", y="MORTGAGE30US", marker='o', label='Loan Count')
plt.axvline(pd.to_datetime('2019-12-01'), color='red', linestyle='--', label='Dec 2019')
plt.title("Mortgage Rate by Loan Age")
plt.xlabel("Loan Age (Months)")
plt.ylabel("Mortgage Rate")

In [ ]:
# Plot refi_benefit_past by loan_age
medians = mort_est_df.groupby('LOAN_AGE')['refi_benefit_past'].median()
p10 = mort_est_df.groupby('LOAN_AGE')['refi_benefit_past'].quantile(0.1)
p90 = mort_est_df.groupby('LOAN_AGE')['refi_benefit_past'].quantile(0.9)
plt.figure(figsize=(10, 6))
plt.plot(medians.index, medians.values, marker='o', linewidth=2, label ='Median')
plt.plot(p10.index, p10.values, linestyle='--', color='orange', label='10th Percentile')
plt.plot(p90.index, p90.values, linestyle='--', color='green', label='90th Percentile')
plt.xlabel('Loan Age')
plt.ylabel('Refinance Benefit Past')
plt.title('Refinance Benefit Past by Loan Age')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Plot current refi_benefit by loan_age
medians = mort_est_df.groupby('LOAN_AGE')['refi_benefit_now'].median()
p10 = mort_est_df.groupby('LOAN_AGE')['refi_benefit_now'].quantile(0.1)
p90 = mort_est_df.groupby('LOAN_AGE')['refi_benefit_now'].quantile(0.9)
plt.figure(figsize=(10, 6))
plt.plot(medians.index, medians.values, marker='o', linewidth=2, label ='Median')
plt.plot(p10.index, p10.values, linestyle='--', color='orange', label='10th Percentile')
plt.plot(p90.index, p90.values, linestyle='--', color='green', label='90th Percentile')
plt.xlabel('Loan Age')
plt.ylabel('Refinance Benefit Now')
plt.title('Refinance Benefit Now by Loan Age')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
ppay_rate =mort_est_df[["LOAN_AGE", "PREPAY_FLAG"]].groupby(['LOAN_AGE']).agg({'PREPAY_FLAG':'mean'}).reset_index()
def_rate = mort_est_df[["LOAN_AGE", "def_flag"]].groupby(['LOAN_AGE']).agg({'def_flag':'mean'}).reset_index()
mod_rate = mort_est_df[["LOAN_AGE", "fb_mod_flag"]].groupby(['LOAN_AGE']).agg({'fb_mod_flag':'mean'}).reset_index()

ppay_rate.rename(columns={'PREPAY_FLAG':'Prepay_Rate'}, inplace=True)
def_rate.rename(columns={'def_flag':'Default_Rate'}, inplace=True)
mod_rate.rename(columns={'fb_mod_flag':'Modification_Rate'}, inplace=True)

ppay_rate = ppay_rate.merge(def_rate, on='LOAN_AGE', how='left').merge(mod_rate, on='LOAN_AGE', how='left')

plt.figure(figsize=(10, 6))
sns.lineplot(data=ppay_rate, x="LOAN_AGE", y="Prepay_Rate", marker='o', label='Prepay Rate')
plt.title("Monthly Prepayment Rates by Loan Age")
plt.xlabel("Loan Age (Months)")
plt.ylabel("Prepay Rate")

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=ppay_rate, x="LOAN_AGE", y="Default_Rate", marker='o', label='Default Rate')
sns.lineplot(data=ppay_rate, x="LOAN_AGE", y="Modification_Rate", marker='o', label='Modification Rate')
plt.title("Monthly Default and Modification Rates by Loan Age")
plt.xlabel("Loan Age (Months)")
plt.ylabel("Monthly Default and Modification Rate")

In [ ]:
#Heatmap of SATO by present and past refi beneift

heatmap_df = mort_est_df[['PREPAY_FLAG', 'refi_benefit_past', 'refi_benefit_now']].copy()


heatmap_df['past_bin'] = pd.cut(heatmap_df['refi_benefit_past'], bins=range(0, 50000, 10000))
heatmap_df['now_bin'] = pd.cut(heatmap_df['refi_benefit_now'], bins=range(-10000, 50000, 10000))

# 3. Create a pivot table 
# We calculate the mean SATO for each combination of bins
heatmap_data = heatmap_df.pivot_table(
    index='past_bin', 
    columns='now_bin', 
    values='PREPAY_FLAG', 
    aggfunc='mean'
)

# 4. Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_data, 
    annot=True,     # Shows the rate values in the cells
    fmt = ".2%",      # Rounds values to 2 decimal places
    cmap='YlOrRd',  # Color map (Yellow to Red)
    cbar_kws={'label': 'Average SATO'}
)

plt.title('Heatmap of Prepayment Rate by Past and Current Refinance Benefit')
plt.xlabel('Current Refinance Benefit')
plt.ylabel('Past Refinance Benefit')
plt.gca().invert_yaxis() # Often preferred to have higher LTV at the top
plt.show()